# 03 Feature Engineering

This notebook builds the feature sets for the binary PII detection models.

The main feature sets are:

1. TF-IDF features
2. Basic text features
3. Privacy-pattern regex features
4. Combined features

I am avoiding any leakage fields, including token labels, privacy masks, or anything created from the answer labels.

## 1. Setup

Install and import the packages needed for feature engineering.

In [1]:
# install needed packages
!pip install datasets pandas pyarrow scikit-learn scipy

In [2]:
# import packages
import pandas as pd
import numpy as np
import re
import string

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer

## 2. Small sample data check

Use a tiny sample first to make sure the feature ideas work before running them on the full dataset.

In [3]:
# create a tiny sample dataset for testing
sample_df = pd.DataFrame({
    "text": [
        "Can you help me write a professional email?",
        "My email is aura@example.com and my phone number is 555-123-4567",
        "Summarize this paragraph for class",
        "The server IP is 192.168.1.10 and the user id is ABC12345",
        "Here is my address: 123 Main Street, Seattle, WA 98101"
    ],
    "label": [0, 1, 0, 1, 1]
})

# preview the sample data
sample_df

,text,label
0,Can you help me write a professional email?,0
1,My email is aura@example.com and my phone numb...,1
2,Summarize this paragraph for class,0
3,The server IP is 192.168.1.10 and the user id ...,1
4,"Here is my address: 123 Main Street, Seattle, ...",1


## 3. TF-IDF sample features

TF-IDF turns text into numeric word and phrase features.

For the real project, TF-IDF should be fit on training text only, then used to transform validation and test text.

In [4]:
# separate sample text and labels
sample_text = sample_df["text"]
sample_labels = sample_df["label"]

# create the tf-idf vectorizer
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1
)

# fit and transform the sample text
X_sample_tfidf = tfidf_vectorizer.fit_transform(sample_text)

# check the shape
print("tf-idf matrix shape:", X_sample_tfidf.shape)

tf-idf matrix shape: (5, 83)


In [5]:
# show the first 25 tf-idf feature names
feature_names = tfidf_vectorizer.get_feature_names_out()

feature_names[:25]

array(['10', '10 and', '123', '123 4567', '123 main', '168', '168 10',
       '192', '192 168', '4567', '555', '555 123', '98101', 'abc12345',
       'address', 'address 123', 'and', 'and my', 'and the', 'aura',
       'aura example', 'can', 'can you', 'class', 'com'], dtype=object)

## 4. Basic text features

These features describe the shape of the text, like length, word count, digits, and punctuation.

In [6]:
# create a function for basic text features
def extract_basic_text_features(texts):
    # store one feature row per text
    rows = []

    # loop through each prompt
    for text in texts:
        # handle missing values safely
        text = "" if pd.isna(text) else str(text)

        # count basic text patterns
        char_count = len(text)
        word_count = len(text.split())
        digit_count = sum(char.isdigit() for char in text)
        uppercase_count = sum(char.isupper() for char in text)
        punctuation_count = sum(char in string.punctuation for char in text)
        whitespace_count = sum(char.isspace() for char in text)

        # check for html-like text
        has_html_like_text = int(bool(re.search(r"<[^>]+>|&(?:amp|lt|gt|quot|apos|nbsp);", text)))

        # add feature row
        rows.append([
            char_count,
            word_count,
            digit_count,
            uppercase_count,
            punctuation_count,
            whitespace_count,
            has_html_like_text
        ])

    # return numeric feature matrix
    return np.array(rows)

In [7]:
# create basic text features for the sample prompts
X_sample_basic = extract_basic_text_features(sample_df["text"])

# name the basic text feature columns
basic_feature_names = [
    "char_count",
    "word_count",
    "digit_count",
    "uppercase_count",
    "punctuation_count",
    "whitespace_count",
    "has_html_like_text"
]

# turn the feature matrix into a dataframe so it is easier to read
basic_features_df = pd.DataFrame(
    X_sample_basic,
    columns=basic_feature_names
)

# preview the basic text features
basic_features_df

,char_count,word_count,digit_count,uppercase_count,punctuation_count,whitespace_count,has_html_like_text
0,43,8,0,1,1,7,0
1,64,10,10,1,4,9,0
2,34,5,0,1,0,4,0
3,57,11,14,6,3,10,0
4,54,10,8,6,3,9,0


## 5. Privacy-pattern regex features

These features look for simple privacy-related patterns like emails, phone numbers, URLs, IP addresses, dates, and address words.

These are not labels. They are just extra signals for the model.

In [8]:
# create regex patterns for privacy-style features
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
phone_pattern = r"(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{3}\)?[\s.-]?)\d{3}[\s.-]?\d{4}"
url_pattern = r"\b(?:https?://|www\.)\S+\b"
ip_pattern = r"\b(?:\d{1,3}\.){3}\d{1,3}\b"
long_number_pattern = r"\b\d[\d\s.-]{7,}\d\b"
date_pattern = r"\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{4}[/-]\d{1,2}[/-]\d{1,2})\b"
address_word_pattern = r"\b(?:street|st\.?|avenue|ave\.?|road|rd\.?|drive|dr\.?|apt|apartment|suite|zip|postal)\b"
id_word_pattern = r"\b(?:ssn|social security|passport|driver'?s license|account number|student id|user id|customer id)\b"

In [9]:
# create a function for privacy-pattern regex features
def extract_privacy_pattern_features(texts):
    # store one feature row per text
    rows = []

    # loop through each prompt
    for text in texts:
        # handle missing values safely
        text = "" if pd.isna(text) else str(text)

        # check whether each privacy pattern appears
        has_email = int(bool(re.search(email_pattern, text)))
        has_phone = int(bool(re.search(phone_pattern, text)))
        has_url = int(bool(re.search(url_pattern, text, flags=re.IGNORECASE)))
        has_ip_address = int(bool(re.search(ip_pattern, text)))
        has_long_number = int(bool(re.search(long_number_pattern, text)))
        has_date_like_pattern = int(bool(re.search(date_pattern, text, flags=re.IGNORECASE)))
        has_address_word = int(bool(re.search(address_word_pattern, text, flags=re.IGNORECASE)))
        has_id_word = int(bool(re.search(id_word_pattern, text, flags=re.IGNORECASE)))

        # add feature row
        rows.append([
            has_email,
            has_phone,
            has_url,
            has_ip_address,
            has_long_number,
            has_date_like_pattern,
            has_address_word,
            has_id_word
        ])

    # return numeric feature matrix
    return np.array(rows)

In [10]:
# create privacy-pattern features for the sample prompts
X_sample_privacy = extract_privacy_pattern_features(sample_df["text"])

# name the privacy-pattern feature columns
privacy_feature_names = [
    "has_email",
    "has_phone",
    "has_url",
    "has_ip_address",
    "has_long_number",
    "has_date_like_pattern",
    "has_address_word",
    "has_id_word"
]

# turn the feature matrix into a dataframe so it is easier to read
privacy_features_df = pd.DataFrame(
    X_sample_privacy,
    columns=privacy_feature_names
)

# preview the privacy-pattern features
privacy_features_df

,has_email,has_phone,has_url,has_ip_address,has_long_number,has_date_like_pattern,has_address_word,has_id_word
0,0,0,0,0,0,0,0,0
1,1,1,0,0,1,0,0,0
2,0,0,0,0,0,0,0,0
3,0,0,0,1,1,0,0,1
4,0,0,0,0,0,0,1,0


## 6. Combined sample features

Combine TF-IDF, text features, and pattern features into one feature matrix.

In [11]:
# import sparse matrix helper
from scipy.sparse import csr_matrix

In [12]:
# convert basic and privacy features to sparse matrices
X_sample_basic_sparse = csr_matrix(X_sample_basic)
X_sample_privacy_sparse = csr_matrix(X_sample_privacy)

# combine tf-idf, basic text, and privacy-pattern features
X_sample_combined = hstack([
    X_sample_tfidf,
    X_sample_basic_sparse,
    X_sample_privacy_sparse
])

# check combined feature shape
print("tf-idf shape:", X_sample_tfidf.shape)
print("basic text shape:", X_sample_basic_sparse.shape)
print("privacy pattern shape:", X_sample_privacy_sparse.shape)
print("combined shape:", X_sample_combined.shape)

tf-idf shape: (5, 83)
basic text shape: (5, 7)
privacy pattern shape: (5, 8)
combined shape: (5, 98)


## 7. Sample feature check summary

The small sample check worked.

It created:

1. TF-IDF features
2. Basic text features
3. Privacy-pattern regex features
4. Combined features

The regex features are simple and imperfect, but that is okay. They are just signals for the model, not final decisions.

The next sections use the actual repo files and real train, validation, and test splits.

## 8. Real project feature pipeline

Now switch from the small sample to the real project data.

The split files are created by `src/data/data_split.py`.

The repo commits `split_metadata.json`, but the actual parquet split files are created locally and ignored by GitHub.

### 8.1 Load the GitHub repo into Colab

Colab opens the notebook, but it does not automatically load the whole repo.

This step clones the repo so the notebook can find `src/`, `data_splits/`, and the feature files.

In [13]:
# # bring in pathlib so we can check whether folders already exist
# from pathlib import Path

# # move to the main colab working folder
# %cd /content

# # save the github repo link so we only have to write it once
# repo_url = "https://github.com/Cyber-207/cyber207_specialized_PII_detection_comparison.git"

# # save the folder name that git will create when it clones the repo
# repo_name = "cyber207_specialized_PII_detection_comparison"

# # check whether the repo folder is already in this colab session
# if not Path(repo_name).exists():
#     # clone the repo if it is not already here
#     !git clone {repo_url}
# else:
#     # skip cloning if the repo is already here
#     print("Repo already cloned.")

# # move into the repo folder so later paths work correctly
# %cd /content/{repo_name}

In [14]:
# check that colab is inside the project repo

# import tools for checking folders and file paths
import os
from pathlib import Path

# save the current folder as the repo root
repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
print(repo_root)

# print where this notebook is currently running
print("Current directory:", repo_root)

# print a label before checking the files we need
print("\nExpected project files:")

# list the repo files this notebook needs before the real feature pipeline can run
expected_paths = [
    repo_root + "/src/data/data_split.py",
    repo_root + "/src/features/text_features.py",
    repo_root + "/src/features/pattern_features.py",
    repo_root + "/data_splits/split_metadata.json",
]

# check each expected file and print whether colab can see it
for path in expected_paths:
    # mark the file as found or missing
    status = "FOUND" if Path(path).exists() else "MISSING"

    # print the result for this file
    print(f"{path}: {status}")

/home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison
Current directory: /home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison

Expected project files:
/home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison/src/data/data_split.py: FOUND
/home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison/src/features/text_features.py: FOUND
/home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison/src/features/pattern_features.py: FOUND
/home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison/data_splits/split_metadata.json: FOUND


### 8.2 Create or load the frozen split files

Check whether the train, validation, and test parquet files already exist.

If they are missing, run the shared split script to create them.

In [15]:
# create split parquet files if they do not exist yet

from pathlib import Path

# list the split files the feature notebook needs
split_paths = [
    Path(repo_root + "/data_splits/train.parquet"),
    Path(repo_root + "/data_splits/val.parquet"),
    Path(repo_root + "/data_splits/test.parquet"),
]

# use the existing split files if they are already there
if all(path.exists() for path in split_paths):
    print("Split parquet files already exist.")
else:
    # rebuild the split files from the shared data split script
    print("Split parquet files missing. Running data_split.py...")
    !python src/data/data_split.py

Split parquet files already exist.


In [16]:
# confirm the split files were created

for path in split_paths:
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{path}: {status}")

/home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison/data_splits/train.parquet: FOUND
/home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison/data_splits/val.parquet: FOUND
/home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison/data_splits/test.parquet: FOUND


### 8.3 Load the frozen train, validation, and test splits

Load the cleaned train, validation, and test files into dataframes.

In [17]:
# load the cleaned split files into dataframes

# read the local parquet files created by data_split.py
train_df = pd.read_parquet(repo_root + "/data_splits/train.parquet")
val_df = pd.read_parquet(repo_root + "/data_splits/val.parquet")
test_df = pd.read_parquet(repo_root + "/data_splits/test.parquet")

# check the number of rows and columns in each split
print("train shape:", train_df.shape)
print("val shape:", val_df.shape)
print("test shape:", test_df.shape)

train shape: (260338, 3)
val shape: (32542, 3)
test shape: (32543, 3)


In [18]:
# check the columns available in each split

# print the column names so we know what fields we can safely use
print("train columns:", train_df.columns.tolist())
print("val columns:", val_df.columns.tolist())
print("test columns:", test_df.columns.tolist())

train columns: ['text', 'label', 'original_index']
val columns: ['text', 'label', 'original_index']
test columns: ['text', 'label', 'original_index']


In [19]:
# check the label balance in each split

# show the percent of safe and sensitive examples in each split
for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{split_name} label counts")
    print(df["label"].value_counts())

    print(f"\n{split_name} label percentages")
    print(df["label"].value_counts(normalize=True).round(3))


train label counts
label
1    175408
0     84930
Name: count, dtype: int64

train label percentages
label
1    0.674
0    0.326
Name: proportion, dtype: float64

val label counts
label
1    21926
0    10616
Name: count, dtype: int64

val label percentages
label
1    0.674
0    0.326
Name: proportion, dtype: float64

test label counts
label
1    21926
0    10617
Name: count, dtype: int64

test label percentages
label
1    0.674
0    0.326
Name: proportion, dtype: float64


### 8.4 Import feature functions

Use the feature files from the repo instead of the sample functions above.

In [20]:
import sys
# import the feature functions from the repo
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# get the basic text features
from src.features.text_features import extract_text_features

# get the privacy pattern features
from src.features.pattern_features import extract_pattern_features

# make sure the imports worked
print("feature functions imported")

feature functions imported


### 8.5 Quick feature check

Run the repo feature functions on a few training rows first.

In [21]:
# test the repo feature functions on a few real rows

# grab a few rows from the training split
sample_train_text = train_df["text"].head(5)

# make text features for the sample rows
sample_text_features = extract_text_features(sample_train_text)

# make pattern features for the sample rows
sample_pattern_features = extract_pattern_features(sample_train_text)

# check the feature shapes
print("sample text feature shape:", sample_text_features.shape)
print("sample pattern feature shape:", sample_pattern_features.shape)

sample text feature shape: (5, 10)
sample pattern feature shape: (5, 8)


In [22]:
# look at the text features

sample_text_features

,char_count,word_count,digit_count,upper_count,punct_count,whitespace_count,special_char_count,digit_ratio,max_digit_run,has_html
0,101,10,20,4,18,9,9,0.198020,4,1
1,118,16,3,20,12,15,5,0.025424,1,0
2,154,17,12,24,6,16,6,0.077922,5,0
3,199,18,6,20,11,17,7,0.030151,1,1
4,188,23,27,19,20,22,14,0.143617,4,0


In [23]:
# look at the pattern features

sample_pattern_features

,has_email,has_phone,has_url,has_ip,has_long_number,has_date,has_address_words,has_id_words
0,0,0,0,0,0,1,0,0
1,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0
4,1,1,0,1,0,0,0,0


### 8.6 Build TF-IDF features

Fit TF-IDF on the training text only, then use the same vectorizer on validation and test.

In [24]:
# prepare text and labels for the real split

# fill missing text values just in case
train_text = train_df["text"].fillna("")
val_text = val_df["text"].fillna("")
test_text = test_df["text"].fillna("")

# save labels separately for modeling
y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

# check label array shapes
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

y_train shape: (260338,)
y_val shape: (32542,)
y_test shape: (32543,)


In [25]:
# build the tf-idf features

# create the vectorizer for word and short phrase features
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000
)

# fit only on training text so validation and test do not leak into the vocabulary
X_train_tfidf = tfidf_vectorizer.fit_transform(train_text)

# use the training vocabulary to transform validation and test text
X_val_tfidf = tfidf_vectorizer.transform(val_text)
X_test_tfidf = tfidf_vectorizer.transform(test_text)

# check the tf-idf matrix shapes
print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_val_tfidf shape:", X_val_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

X_train_tfidf shape: (260338, 50000)
X_val_tfidf shape: (32542, 50000)
X_test_tfidf shape: (32543, 50000)


### 8.7 Build text and pattern features

Run the approved text and pattern feature functions on the full train, validation, and test splits.

In [26]:
# build the basic text features

# run the approved text feature function on each split
train_text_features = extract_text_features(train_text)
val_text_features = extract_text_features(val_text)
test_text_features = extract_text_features(test_text)

# save the text feature column names
text_feature_columns = train_text_features.columns.tolist()

# check the text feature shapes
print("train text features:", train_text_features.shape)
print("val text features:", val_text_features.shape)
print("test text features:", test_text_features.shape)

# preview the feature columns
text_feature_columns

train text features: (260338, 10)
val text features: (32542, 10)
test text features: (32543, 10)


['char_count',
 'word_count',
 'digit_count',
 'upper_count',
 'punct_count',
 'whitespace_count',
 'special_char_count',
 'digit_ratio',
 'max_digit_run',
 'has_html']

In [27]:
# build the privacy pattern features

# run the approved pattern feature function on each split
train_pattern_features = extract_pattern_features(train_text)
val_pattern_features = extract_pattern_features(val_text)
test_pattern_features = extract_pattern_features(test_text)

# save the pattern feature column names
pattern_feature_columns = train_pattern_features.columns.tolist()

# check the pattern feature shapes
print("train pattern features:", train_pattern_features.shape)
print("val pattern features:", val_pattern_features.shape)
print("test pattern features:", test_pattern_features.shape)

# preview the feature columns
pattern_feature_columns

train pattern features: (260338, 8)
val pattern features: (32542, 8)
test pattern features: (32543, 8)


['has_email',
 'has_phone',
 'has_url',
 'has_ip',
 'has_long_number',
 'has_date',
 'has_address_words',
 'has_id_words']

### 8.8 Combine the feature sets

Combine TF-IDF, text features, and pattern features into one sparse matrix for each split.

In [28]:
# scale text and pattern features before converting to sparse matrices

# import the scaler and sparse matrix helpers
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import csr_matrix, hstack

# create scalers for the numeric feature groups
text_feature_scaler = MinMaxScaler()
pattern_feature_scaler = MinMaxScaler()

# fit the text feature scaler on train only
train_text_scaled = text_feature_scaler.fit_transform(train_text_features)

# use the train-fitted text scaler on validation and test
val_text_scaled = text_feature_scaler.transform(val_text_features)
test_text_scaled = text_feature_scaler.transform(test_text_features)

# fit the pattern feature scaler on train only
train_pattern_scaled = pattern_feature_scaler.fit_transform(train_pattern_features)

# use the train-fitted pattern scaler on validation and test
val_pattern_scaled = pattern_feature_scaler.transform(val_pattern_features)
test_pattern_scaled = pattern_feature_scaler.transform(test_pattern_features)

# convert scaled text features to sparse matrices
X_train_text = csr_matrix(train_text_scaled)
X_val_text = csr_matrix(val_text_scaled)
X_test_text = csr_matrix(test_text_scaled)

# convert scaled pattern features to sparse matrices
X_train_pattern = csr_matrix(train_pattern_scaled)
X_val_pattern = csr_matrix(val_pattern_scaled)
X_test_pattern = csr_matrix(test_pattern_scaled)

# check the scaled sparse feature shapes
print("X_train_text shape:", X_train_text.shape)
print("X_train_pattern shape:", X_train_pattern.shape)

X_train_text shape: (260338, 10)
X_train_pattern shape: (260338, 8)


In [29]:
# combine tf-idf, text, and pattern features

# stack the feature groups side by side
X_train_combined = hstack([X_train_tfidf, X_train_text, X_train_pattern]).tocsr()
X_val_combined = hstack([X_val_tfidf, X_val_text, X_val_pattern]).tocsr()
X_test_combined = hstack([X_test_tfidf, X_test_text, X_test_pattern]).tocsr()

# check the final combined shapes
print("X_train_combined shape:", X_train_combined.shape)
print("X_val_combined shape:", X_val_combined.shape)
print("X_test_combined shape:", X_test_combined.shape)

X_train_combined shape: (260338, 50018)
X_val_combined shape: (32542, 50018)
X_test_combined shape: (32543, 50018)


### 8.9 Save feature files

Save the feature matrices so the modeling notebooks can load them later.

These files are generated locally and should stay out of GitHub.

In [30]:
# save feature matrices and labels for modeling

# import save helpers
import json
import joblib
import numpy as np

from scipy.sparse import save_npz
from pathlib import Path

# create the output folder if it does not already exist
feature_dir = Path("feature_matrices")
feature_dir.mkdir(exist_ok=True)

# save tf-idf matrices
save_npz(feature_dir / "X_train_tfidf.npz", X_train_tfidf)
save_npz(feature_dir / "X_val_tfidf.npz", X_val_tfidf)
save_npz(feature_dir / "X_test_tfidf.npz", X_test_tfidf)

# save text feature matrices
save_npz(feature_dir / "X_train_text.npz", X_train_text)
save_npz(feature_dir / "X_val_text.npz", X_val_text)
save_npz(feature_dir / "X_test_text.npz", X_test_text)

# save pattern feature matrices
save_npz(feature_dir / "X_train_pattern.npz", X_train_pattern)
save_npz(feature_dir / "X_val_pattern.npz", X_val_pattern)
save_npz(feature_dir / "X_test_pattern.npz", X_test_pattern)

# save combined feature matrices
save_npz(feature_dir / "X_train_combined.npz", X_train_combined)
save_npz(feature_dir / "X_val_combined.npz", X_val_combined)
save_npz(feature_dir / "X_test_combined.npz", X_test_combined)

# save labels
np.save(feature_dir / "y_train.npy", y_train)
np.save(feature_dir / "y_val.npy", y_val)
np.save(feature_dir / "y_test.npy", y_test)

# save the tf-idf vectorizer so modeling can reuse the same vocabulary
joblib.dump(tfidf_vectorizer, feature_dir / "tfidf_vectorizer.joblib")

# save the scalers so modeling can reuse the same scaling setup
joblib.dump(text_feature_scaler, feature_dir / "text_feature_scaler.joblib")
joblib.dump(pattern_feature_scaler, feature_dir / "pattern_feature_scaler.joblib")

# save the structured feature column names
with open(feature_dir / "text_feature_columns.json", "w") as f:
    json.dump(text_feature_columns, f, indent=2)

with open(feature_dir / "pattern_feature_columns.json", "w") as f:
    json.dump(pattern_feature_columns, f, indent=2)

print("feature files saved")

feature files saved


In [31]:
# save a small metadata file for the feature outputs

# collect the main settings and shapes
feature_metadata = {
    "tfidf_settings": {
        "lowercase": True,
        "ngram_range": [1, 2],
        "min_df": 2,
        "max_features": 50000,
    },
    "numeric_feature_scaling": {
        "scaler": "MinMaxScaler",
        "fit_on": "train only",
        "applied_to": ["text features", "pattern features"],
    },
    "shapes": {
        "X_train_tfidf": X_train_tfidf.shape,
        "X_val_tfidf": X_val_tfidf.shape,
        "X_test_tfidf": X_test_tfidf.shape,
        "X_train_text": X_train_text.shape,
        "X_val_text": X_val_text.shape,
        "X_test_text": X_test_text.shape,
        "X_train_pattern": X_train_pattern.shape,
        "X_val_pattern": X_val_pattern.shape,
        "X_test_pattern": X_test_pattern.shape,
        "X_train_combined": X_train_combined.shape,
        "X_val_combined": X_val_combined.shape,
        "X_test_combined": X_test_combined.shape,
    },
    "text_feature_columns": text_feature_columns,
    "pattern_feature_columns": pattern_feature_columns,
}

# convert tuple shapes to lists so json can save them
feature_metadata["shapes"] = {
    name: list(shape)
    for name, shape in feature_metadata["shapes"].items()
}

# write the metadata file
with open(feature_dir / "feature_metadata.json", "w") as f:
    json.dump(feature_metadata, f, indent=2)

print("feature metadata saved")

feature metadata saved


### 8.10 Check saved files

Make sure the feature files were written.

In [32]:
# check what was saved in the feature folder
print(Path.cwd())
# list saved files
saved_files = sorted(feature_dir.iterdir())

# print each saved file name
for file in saved_files:
    print(file.name)

/home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison/notebooks
X_test_combined.npz
X_test_pattern.npz
X_test_text.npz
X_test_tfidf.npz
X_train_combined.npz
X_train_pattern.npz
X_train_text.npz
X_train_tfidf.npz
X_val_combined.npz
X_val_pattern.npz
X_val_text.npz
X_val_tfidf.npz
feature_metadata.json
pattern_feature_columns.json
pattern_feature_scaler.joblib
text_feature_columns.json
text_feature_scaler.joblib
tfidf_vectorizer.joblib
y_test.npy
y_train.npy
y_val.npy


### 8.11 Test loading saved files

Check that the saved feature files can be loaded again.

In [33]:
# test loading the saved feature files

# import load helpers
from scipy.sparse import load_npz
import numpy as np
import joblib

# load one combined matrix
loaded_X_train_combined = load_npz(feature_dir / "X_train_combined.npz")

# load one label file
loaded_y_train = np.load(feature_dir / "y_train.npy")

# load the saved tf-idf vectorizer
loaded_tfidf_vectorizer = joblib.load(feature_dir / "tfidf_vectorizer.joblib")

# load the saved scalers
loaded_text_feature_scaler = joblib.load(feature_dir / "text_feature_scaler.joblib")
loaded_pattern_feature_scaler = joblib.load(feature_dir / "pattern_feature_scaler.joblib")

# check that the loaded files have the expected shapes
print("loaded X_train_combined shape:", loaded_X_train_combined.shape)
print("loaded y_train shape:", loaded_y_train.shape)
print("loaded tf-idf vocab size:", len(loaded_tfidf_vectorizer.vocabulary_))
print("loaded text scaler:", type(loaded_text_feature_scaler).__name__)
print("loaded pattern scaler:", type(loaded_pattern_feature_scaler).__name__)

loaded X_train_combined shape: (260338, 50018)
loaded y_train shape: (260338,)
loaded tf-idf vocab size: 50000
loaded text scaler: MinMaxScaler
loaded pattern scaler: MinMaxScaler


### 8.12 Final feature summary

Quick check of the feature sets created by this notebook.

In [34]:
# make a quick feature summary table

# collect the feature matrix shapes
feature_summary = pd.DataFrame({
    "feature_set": [
        "tf-idf",
        "text features",
        "pattern features",
        "combined features",
    ],
    "train_shape": [
        X_train_tfidf.shape,
        X_train_text.shape,
        X_train_pattern.shape,
        X_train_combined.shape,
    ],
    "val_shape": [
        X_val_tfidf.shape,
        X_val_text.shape,
        X_val_pattern.shape,
        X_val_combined.shape,
    ],
    "test_shape": [
        X_test_tfidf.shape,
        X_test_text.shape,
        X_test_pattern.shape,
        X_test_combined.shape,
    ],
})

# show the summary table
feature_summary

,feature_set,train_shape,val_shape,test_shape
0,tf-idf,"(260338, 50000)","(32542, 50000)","(32543, 50000)"
1,text features,"(260338, 10)","(32542, 10)","(32543, 10)"
2,pattern features,"(260338, 8)","(32542, 8)","(32543, 8)"
3,combined features,"(260338, 50018)","(32542, 50018)","(32543, 50018)"


### Done

This notebook creates the feature files for the modeling notebooks.

The saved files are in `feature_matrices/`.

These files are generated locally and should not be committed to GitHub.